<a href="https://colab.research.google.com/github/PARIJAAT-13/Flyrank-A.I/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PARIJAAT-13/Flyrank-A.I/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I chose a Random Forest model because the lane involves ranking content opportunities from multiple decision-time signals. Random Forest can capture nonlinear relationships between signals such as staleness, search volume, impressions, and ranking position without requiring strong linear assumptions.

The model is used as a practical supervised baseline for comparison with the Week-4 rule-based baseline. I will evaluate it on the same data and metric and will interpret feature importance and errors rather than assuming that a more complex model is automatically better.

The goal is decision-support, not ground-truth prediction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Section 1: Method setup

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Method: Random Forest")
print("Purpose: Learn a decision-time priority score for content opportunities.")
print("Baseline: Week-4 rule-based baseline")
print("Evaluation: Same data and decision-time signals.")

Method: Random Forest
Purpose: Learn a decision-time priority score for content opportunities.
Baseline: Week-4 rule-based baseline
Evaluation: Same data and decision-time signals.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a grouped split by client_id so that content from the same client does not appear in both training and test sets.

This is an honest validation design because the model should be evaluated on clients it did not see during training. It reduces the risk of client-specific information leaking across the split.

The same decision-time signals used by the Week-4 baseline will be used for the model comparison.

In [ ]:
# Reconnect the FlyRank repository if the Colab runtime was reset

import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/PARIJAAT-13/Flyrank-A.I.git"
REPO_DIR = Path("/content/Flyrank-A.I")

if not REPO_DIR.exists():
    print("Repository not found. Cloning...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )
else:
    print("Repository already exists.")

DATA_PATH = REPO_DIR / "data/raw/content_refresh_anonymized.csv"

print("\nRepository exists:", REPO_DIR.exists())
print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)

if not DATA_PATH.exists():
    print("\n❌ Dataset not found.")
    print("Checking data/raw:")
    if (REPO_DIR / "data/raw").exists():
        print(os.listdir(REPO_DIR / "data/raw"))
else:
    print("\n✅ Dataset found!")

Repository already exists.

Repository exists: True
Dataset exists: True
Dataset path: /content/Flyrank-A.I/data/raw/content_refresh_anonymized.csv

✅ Dataset found!


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Section 2: Grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

# Load the same starter dataset used for the Week-4 baseline
repo_root = Path("/content/Flyrank-A.I")
data_path = repo_root / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print(f"Dataset rows: {len(df):,}")
print(f"Unique clients: {df['client_id'].nunique():,}")

# Decision-time features only
feature_cols = [
    "days_since_last_update",
    "search_volume",
    "impressions_last_30d",
    "avg_position"
]

# Convert features to numeric
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Create the same baseline-style score as the Week-4 decision rule.
df["staleness_score"] = df["days_since_last_update"].rank(pct=True)
df["volume_score"] = df["search_volume"].rank(pct=True)
df["impression_score"] = df["impressions_last_30d"].rank(pct=True)

df["position_score"] = (
    (df["avg_position"] >= 4) &
    (df["avg_position"] <= 20)
).astype(float)

df["baseline_score"] = (
    0.40 * df["staleness_score"]
    + 0.30 * df["volume_score"]
    + 0.20 * df["impression_score"]
    + 0.10 * df["position_score"]
)

# Grouped split: entire clients stay in one split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        groups=df["client_id"]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

overlap = train_clients.intersection(test_clients)

print("\n=== SPLIT SUMMARY ===")
print(f"Training rows: {len(train_df):,}")
print(f"Test rows:     {len(test_df):,}")
print(f"Training clients: {len(train_clients):,}")
print(f"Test clients:     {len(test_clients):,}")
print(f"Client overlap: {len(overlap)}")

if len(overlap) == 0:
    print("✅ Grouped split passed: no client appears in both train and test.")
else:
    print("❌ WARNING: client overlap detected.")

Dataset rows: 30,000
Unique clients: 32

=== SPLIT SUMMARY ===
Training rows: 23,837
Test rows:     6,163
Training clients: 25
Test clients:     7
Client overlap: 0
✅ Grouped split passed: no client appears in both train and test.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Train and compare against my baseline

I will train the Random Forest using the same decision-time signals used by the Week-4 baseline.

The model will be evaluated on the held-out test set created by the grouped client split. The Week-4 baseline score will be evaluated on exactly the same test rows using the same MAE metric.

A lower MAE indicates that the predictions are closer to the baseline score. The comparison is used to measure whether the model provides a useful improvement over the simple baseline, rather than assuming that model complexity is automatically better.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Section 3: Train Random Forest and compare with Week-4 baseline

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Features: decision-time signals only
X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

# Target:
# We use the Week-4 baseline score as the reference target.
y_train = train_df["baseline_score"].copy()
y_test = test_df["baseline_score"].copy()

# Train Random Forest
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Model predictions
model_pred = model.predict(X_test)

# Week-4 baseline predictions
baseline_pred = test_df["baseline_score"].values

# Calculate MAE
model_mae = mean_absolute_error(y_test, model_pred)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

# Calculate RMSE as an additional diagnostic
model_rmse = np.sqrt(mean_squared_error(y_test, model_pred))
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

# Comparison table
comparison = pd.DataFrame({
    "method": [
        "Week-4 Baseline",
        "Random Forest"
    ],
    "MAE": [
        baseline_mae,
        model_mae
    ],
    "RMSE": [
        baseline_rmse,
        model_rmse
    ]
})

print("=== MODEL VS BASELINE ===")
display(comparison)

# Improvement relative to baseline
if baseline_mae != 0:
    mae_improvement = (
        (baseline_mae - model_mae) /
        baseline_mae
    ) * 100
else:
    mae_improvement = 0

print(f"Baseline MAE:       {baseline_mae:.6f}")
print(f"Random Forest MAE:  {model_mae:.6f}")
print(f"MAE change:         {mae_improvement:.2f}%")

if model_mae < baseline_mae:
    print("✅ Random Forest has lower MAE than the baseline.")
elif model_mae > baseline_mae:
    print("ℹ️ Random Forest has higher MAE than the baseline.")
else:
    print("ℹ️ Random Forest and baseline have the same MAE.")

# Feature importance
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("\n=== FEATURE IMPORTANCE ===")
display(importance)


=== MODEL VS BASELINE ===


,method,MAE,RMSE
0,Week-4 Baseline,0.000000,0.000000
1,Random Forest,0.013318,0.024344


Baseline MAE:       0.000000
Random Forest MAE:  0.013318
MAE change:         0.00%
ℹ️ Random Forest has higher MAE than the baseline.

=== FEATURE IMPORTANCE ===


,feature,importance
0,days_since_last_update,0.582121
1,search_volume,0.191878
2,impressions_last_30d,0.139977
3,avg_position,0.086025


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

I compared the Random Forest predictions with the Week-4 baseline reference on the held-out test set.

The largest errors are cases where the model's predicted priority differs substantially from the baseline score. These cases are useful for understanding where the model may be less reliable.

I also inspect feature importance to understand which decision-time signals the model relies on most. Feature importance shows association within this model and should not be interpreted as causal evidence.

The model should therefore be treated as decision-support rather than a ground-truth predictor.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Section 4: Error analysis and interpretation

# Create an error-analysis table
error_analysis = test_df[
    ["content_id", "client_id"] + feature_cols + ["baseline_score"]
].copy()

error_analysis["model_prediction"] = model_pred

# Absolute prediction error
error_analysis["absolute_error"] = (
    error_analysis["model_prediction"] -
    error_analysis["baseline_score"]
).abs()

# Largest errors first
largest_errors = error_analysis.sort_values(
    "absolute_error",
    ascending=False
).head(10)

print("=== TOP 10 LARGEST ERRORS ===")

display(
    largest_errors[
        [
            "content_id",
            "client_id",
            "baseline_score",
            "model_prediction",
            "absolute_error",
            "days_since_last_update",
            "search_volume",
            "impressions_last_30d",
            "avg_position"
        ]
    ]
)

# Feature importance
print("\n=== FEATURE IMPORTANCE ===")

display(importance)

# Error summary
print("\n=== ERROR SUMMARY ===")
print(f"Mean absolute error: {model_mae:.6f}")
print(f"Root mean squared error: {model_rmse:.6f}")

# Compare prediction direction
error_analysis["prediction_difference"] = (
    error_analysis["model_prediction"] -
    error_analysis["baseline_score"]
)

over_predictions = (
    error_analysis["prediction_difference"] > 0
).sum()

under_predictions = (
    error_analysis["prediction_difference"] < 0
).sum()

print(f"Predictions above baseline: {over_predictions:,}")
print(f"Predictions below baseline: {under_predictions:,}")

# Simple interpretation
top_feature = importance.iloc[0]["feature"]

print("\n=== INTERPRETATION ===")
print(
    f"The most important feature in the Random Forest was "
    f"'{top_feature}'."
)

print(
    "The largest errors show cases where the learned model "
    "differs most from the Week-4 baseline."
)

print(
    "These errors should be reviewed before using the model "
    "for operational decisions."
)


=== TOP 10 LARGEST ERRORS ===


,content_id,client_id,baseline_score,model_prediction,absolute_error,days_since_last_update,search_volume,impressions_last_30d,avg_position
27271,content_7bc32bc1df59,client_8527a891e2,0.504635,0.651301,0.146666,92,20.0,0,0.0
6104,content_f1743a78cb16,client_8527a891e2,0.504635,0.651301,0.146666,92,20.0,0,2.0
2910,content_6560653b173e,client_8527a891e2,0.506928,0.651301,0.144373,103,20.0,0,3.2
19205,content_17c2de35424d,client_8527a891e2,0.506928,0.651301,0.144373,103,20.0,0,1.0
29551,content_93e94f1ff1f5,client_4e07408562,0.349283,0.491267,0.141984,7,10.0,1835,3.7
19787,content_3caffc05c7f0,client_4e07408562,0.407628,0.549166,0.141538,7,140.0,330,2.0
28621,content_8acc02dd9cad,client_4e07408562,0.363608,0.504390,0.140782,7,20.0,468,3.1
9392,content_a05de90216c2,client_4e07408562,0.431605,0.566661,0.135056,7,90.0,1031,3.8
12535,content_e9e556620469,client_4e07408562,0.417410,0.551486,0.134076,7,210.0,376,3.9
8876,content_4092ad6d1f71,client_4e07408562,0.421002,0.553875,0.132874,7,70.0,840,3.8



=== FEATURE IMPORTANCE ===


,feature,importance
0,days_since_last_update,0.582121
1,search_volume,0.191878
2,impressions_last_30d,0.139977
3,avg_position,0.086025



=== ERROR SUMMARY ===
Mean absolute error: 0.013318
Root mean squared error: 0.024344
Predictions above baseline: 3,230
Predictions below baseline: 2,933

=== INTERPRETATION ===
The most important feature in the Random Forest was 'days_since_last_update'.
The largest errors show cases where the learned model differs most from the Week-4 baseline.
These errors should be reviewed before using the model for operational decisions.




## Self-check

- [x] Every section is filled with both reasoning and supporting code.
- [x] The notebook runs top to bottom without errors.
- [x] The model uses only decision-time signals.
- [x] The train/test split is grouped by client_id with no client overlap.
- [x] The Random Forest is compared with the Week-4 baseline on the same held-out test set.
- [x] MAE and RMSE are reported.
- [x] Feature importance and the largest errors are reviewed.
- [x] The results are described as directional decision-support, not ground truth.
- [x] No client names, URLs, or private queries are included in the notebook output.
- [x] The notebook is saved as `work/notebooks/w05_model.ipynb` and will be committed to the repository.